In [1]:
import pandas as pd
import json
from pathlib import Path
from sklearn.metrics import mean_absolute_error, accuracy_score, f1_score, cohen_kappa_score

def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def normalize_gold(gold_path):
    rows = []
    for obj in read_jsonl(gold_path):
        rows.append(
            {
                "article_id": obj.get("article_id"),
                "gold_rating": obj.get("rating"),
                "gold_comment": obj.get("comment"),
            }
        )
    return pd.DataFrame(rows)


def normalize_pred(pred_path):
    rows = []
    for obj in read_jsonl(pred_path):
        rows.append(
            {
                "article_id": obj.get("article_id"),
                "llm_rating": obj.get("rating"),
                "llm_comment": obj.get("comment"),
                "status": obj.get("status", "success"),
            }
        )
    df = pd.DataFrame(rows)
    if "status" in df.columns:
        df = df[df["status"] == "success"].copy()
    return df


def infer_dataset_name(folder_path):
    return Path(folder_path).name


def infer_model_name(pred_file_name):
    name = pred_file_name.replace(".jsonl", "")
    if name.startswith("gemini"):
        return "gemini"
    if name.startswith("qwen"):
        return "qwen"
    if name.startswith("claude"):
        return "claude"
    if name.startswith("gpt"):
        return "gpt"
    return name


def map_to_3_class(rating):
    """Collapses 5-point scale into 3-point scale (Negative, Neutral, Positive)"""
    if rating in [-2, -1]:
        return -1
    elif rating == 0:
        return 0
    elif rating in [1, 2]:
        return 1
    return rating


def evaluate_one_pair(gold_path, pred_path):
    df_gold = normalize_gold(gold_path)
    df_pred = normalize_pred(pred_path)

    df = pd.merge(df_gold, df_pred, on="article_id", how="inner")

    total_merged = len(df)
    df = df.dropna(subset=["gold_rating", "llm_rating"]).copy()
    valid_rows = len(df)

    if valid_rows == 0:
        return None, None

    # Cast to integer for metric calculations
    df["gold_rating"] = df["gold_rating"].astype(int)
    df["llm_rating"] = df["llm_rating"].astype(int)
    
    # Map to 3-class for polarity evaluation
    df["gold_rating_3c"] = df["gold_rating"].apply(map_to_3_class)
    df["llm_rating_3c"] = df["llm_rating"].apply(map_to_3_class)

    labels_5c = [-2, -1, 0, 1, 2]
    labels_3c = [-1, 0, 1]

    metrics = {
        "model": infer_model_name(Path(pred_path).name),
        "pred_file": Path(pred_path).name,
        "valid_rows": valid_rows,
        "coverage": round(valid_rows / max(len(df_gold), 1), 4),
        
        # Five-Class Evaluation (Intensity)
        "qwk_5": round(cohen_kappa_score(df["gold_rating"], df["llm_rating"], weights="quadratic", labels=labels_5c), 4),
        "mae_5": round(mean_absolute_error(df["gold_rating"], df["llm_rating"]), 4),
        "acc_5": round(accuracy_score(df["gold_rating"], df["llm_rating"]), 4),
        
        # Three-Class Evaluation (Polarity)
        "macro_f1_3": round(f1_score(df["gold_rating_3c"], df["llm_rating_3c"], labels=labels_3c, average="macro", zero_division=0), 4),
        "acc_3": round(accuracy_score(df["gold_rating_3c"], df["llm_rating_3c"]), 4),
    }

    return metrics, df


def evaluate_all_models(gold_file, output_csv):
    base = Path.cwd()
    gold_path = (base / gold_file).resolve()

    if not gold_path.exists():
        raise FileNotFoundError(f"Gold file not found: {gold_path}")

    pred_files = sorted(
        p for p in base.glob("*.jsonl")
        if p.name != gold_path.name and not p.name.startswith("gold_standard")
    )

    if not pred_files:
        raise ValueError("No prediction JSONL files found for evaluation.")

    rows = []
    for pred_file in pred_files:
        metrics, _ = evaluate_one_pair(gold_path, pred_file.resolve())
        if metrics is not None:
            rows.append(metrics)

    if not rows:
        raise ValueError("No valid model evaluations were produced.")

    # Sort hierarchy: QWK (Primary 5c), Macro-F1 (Primary 3c), MAE (Secondary 5c)
    summary_df = pd.DataFrame(rows).sort_values(
        ["qwk_5", "macro_f1_3", "mae_5"], ascending=[False, False, True]
    ).reset_index(drop=True)

    summary_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"Saved: {output_csv}")

    return summary_df

if __name__ == "__main__":
    summary_df = evaluate_all_models(
        gold_file="gold_standard_pakistan.jsonl",
        output_csv="pakistan_model_evaluation_summary.csv",
    )

    print(summary_df.head(30))

Saved: pakistan_model_evaluation_summary.csv
    model                              pred_file  valid_rows  coverage  \
0  gemini  gemini3.1_pro_extended_pakistan.jsonl          41    1.0000   
1  claude          claude_sonnet5_pakistan.jsonl          41    1.0000   
2     gpt                   gpt4o_pakistan.jsonl          41    1.0000   
3    qwen              qwen2.5_7b_pakistan.jsonl          38    0.9268   

    qwk_5   mae_5   acc_5  macro_f1_3   acc_3  
0  0.9358  0.0732  0.9268      0.8828  0.9756  
1  0.7525  0.4146  0.6098      0.6573  0.7317  
2  0.6790  0.4390  0.6098      0.6265  0.7073  
3  0.1955  0.8947  0.3684      0.4441  0.4211  
